# PART-2 Paraphrase Detection 실행 기록

이 노트북은 기존 `paraphrase_detection.py`, `datasets.py`, `evaluation.py` 소스를 import해서 Quora paraphrase detection 실행 결과를 한 곳에 기록하기 위한 용도입니다.

- 체크포인트는 프로젝트 루트의 기존 경로를 사용합니다. 기본값은 `10-1e-05-paraphrase.pt`입니다.
- 예측 결과는 제출 코드와 같은 `predictions/para-dev-output.csv`, `predictions/para-test-output.csv`에 기록됩니다.
- 기본 실행 모드는 기존 체크포인트를 로드해서 dev/test prediction을 다시 생성하는 방식입니다.
- 학습까지 새로 돌릴 때는 설정 셀에서 `RUN_TRAIN = True`로 바꿉니다.


## 1. 프로젝트 루트 준비

노트북을 프로젝트 루트에서 열거나 `notebooks/` 폴더 안에서 열어도 기존 Python 소스들을 import할 수 있도록 작업 디렉터리와 `sys.path`를 맞춥니다.

In [ ]:
from pathlib import Path
import os
import sys
import csv
import json
from collections import Counter
from datetime import datetime

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "paraphrase_detection.py").exists():
    candidate = PROJECT_ROOT.parent
    if (candidate / "paraphrase_detection.py").exists():
        PROJECT_ROOT = candidate

assert (PROJECT_ROOT / "paraphrase_detection.py").exists(), (
    "프로젝트 루트 또는 notebooks/ 폴더에서 이 노트북을 실행하세요."
)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


## 2. 기존 소스 코드 불러오기

`paraphrase_detection.py`의 `train()`과 `test()`를 그대로 호출합니다. 노트북에는 실행 설정과 결과 기록만 두고, 모델/데이터셋/평가 로직은 기존 `.py` 파일에 있는 구현을 사용합니다.

In [ ]:
from argparse import Namespace

import torch

from paraphrase_detection import add_arguments, seed_everything, train, test
from datasets import load_paraphrase_data

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")


## 3. 실행 설정

아래 값들은 `paraphrase_detection.py`의 CLI 기본값과 맞춰져 있습니다. 현재 체크포인트로 결과만 다시 쓰려면 `RUN_TRAIN = False`, `RUN_TEST = True`를 유지합니다. GPU가 없는 환경에서 CPU 실행을 강제로 허용하려면 `ALLOW_CPU_EXECUTION = True`로 바꾸면 됩니다.

In [ ]:
RUN_TRAIN = False
RUN_TEST = True
ALLOW_CPU_EXECUTION = False

CHECKPOINT_PATH = None  # None이면 script와 같은 규칙으로 checkpoint 이름을 만듭니다.

args = Namespace(
    para_train="data/quora-train.csv",
    para_dev="data/quora-dev.csv",
    para_test="data/quora-test-student.csv",
    para_dev_out="predictions/para-dev-output.csv",
    para_test_out="predictions/para-test-output.csv",
    seed=11711,
    epochs=10,
    use_gpu=torch.cuda.is_available(),
    bidirectional_eval=False,
    prompt_template="paraphrase",
    prompt_ensemble_eval=False,
    prompt_ensemble_templates="paraphrase,duplicate,equivalent",
    threshold=None,
    tune_threshold=False,
    skip_train=not RUN_TRAIN,
    override_checkpoint_eval_args=False,
    augment_swap=False,
    hard_negative_weight=1.0,
    hard_negative_jaccard=0.6,
    class_weighting=False,
    classification_head=False,
    hidden_dropout_prob=0.1,
    grad_clip=0.0,
    weight_decay=0.0,
    early_stopping_patience=0,
    batch_size=8,
    lr=1e-5,
    model_size="gpt2",
)

def build_checkpoint_path(run_args):
    parts = [str(run_args.epochs), str(run_args.lr)]
    if run_args.classification_head:
        parts.append("linear")
    if run_args.augment_swap:
        parts.append("swap")
    if run_args.hard_negative_weight > 1.0:
        parts.append("hardneg")
    if run_args.class_weighting:
        parts.append("classweight")
    return "-".join(parts + ["paraphrase.pt"])

args.filepath = CHECKPOINT_PATH or build_checkpoint_path(args)
args = add_arguments(args)
seed_everything(args.seed)

config_summary = {
    "run_train": RUN_TRAIN,
    "run_test": RUN_TEST,
    "device": "cuda" if args.use_gpu else "cpu",
    "allow_cpu_execution": ALLOW_CPU_EXECUTION,
    "checkpoint": args.filepath,
    "dev_output": args.para_dev_out,
    "test_output": args.para_test_out,
    "seed": args.seed,
    "epochs": args.epochs,
    "batch_size": args.batch_size,
    "learning_rate": args.lr,
    "model_size": args.model_size,
}
print(json.dumps(config_summary, indent=2, ensure_ascii=False))


## 4. 데이터와 산출물 확인

학습/평가를 시작하기 전에 Quora 데이터 파일, 체크포인트, 기존 prediction 파일이 예상 위치에 있는지 확인합니다. 이 셀은 모델을 실행하지 않고 파일 상태만 기록합니다.

In [ ]:
def file_status(path):
    path = PROJECT_ROOT / path
    if not path.exists():
        return {"path": str(path.relative_to(PROJECT_ROOT)), "exists": False}
    return {
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": True,
        "size_mb": round(path.stat().st_size / (1024 * 1024), 3),
        "modified": datetime.fromtimestamp(path.stat().st_mtime).isoformat(timespec="seconds"),
    }

artifact_status = [
    file_status(args.para_train),
    file_status(args.para_dev),
    file_status(args.para_test),
    file_status(args.filepath),
    file_status(args.para_dev_out),
    file_status(args.para_test_out),
]
print(json.dumps(artifact_status, indent=2, ensure_ascii=False))


## 5. 데이터 로딩 기록

기존 `load_paraphrase_data()` 함수를 사용해서 train/dev/test split을 읽고, 레이블 분포를 간단히 확인합니다. 이 단계도 모델 forward pass는 수행하지 않습니다.

In [ ]:
train_data = load_paraphrase_data(args.para_train)
dev_data = load_paraphrase_data(args.para_dev)
test_data = load_paraphrase_data(args.para_test, split="test")

label_counts = {
    "train": dict(Counter(example[2] for example in train_data)),
    "dev": dict(Counter(example[2] for example in dev_data)),
}
data_summary = {
    "train_examples": len(train_data),
    "dev_examples": len(dev_data),
    "test_examples": len(test_data),
    "label_counts": label_counts,
}
print(json.dumps(data_summary, indent=2, ensure_ascii=False))


## 6. 기존 prediction 파일 요약

이미 만들어진 `predictions/para-*.csv` 파일의 행 수와 예측 라벨 분포를 확인합니다. 모델을 다시 실행하기 전후의 결과가 같은지 비교할 때 이 셀의 출력이 기준 기록이 됩니다.

In [ ]:
def summarize_prediction_file(path, preview_rows=5):
    path = PROJECT_ROOT / path
    if not path.exists():
        return {"path": str(path.relative_to(PROJECT_ROOT)), "exists": False}

    label_counts = Counter()
    preview = []
    total_rows = 0
    with path.open(newline="") as fp:
        reader = csv.DictReader(fp)
        for row in reader:
            total_rows += 1
            label_counts[row.get("Predicted_Is_Paraphrase", "")] += 1
            if len(preview) < preview_rows:
                preview.append(row)

    return {
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": True,
        "rows": total_rows,
        "label_counts": dict(label_counts),
        "preview": preview,
    }

prediction_summary_before = {
    "recorded_at": datetime.now().isoformat(timespec="seconds"),
    "dev": summarize_prediction_file(args.para_dev_out),
    "test": summarize_prediction_file(args.para_test_out),
}
print(json.dumps(prediction_summary_before, indent=2, ensure_ascii=False))


## 7. 학습 및 평가 실행

이 셀에서 기존 `train(args)`와 `test(args)`를 호출합니다. `RUN_TRAIN = False`이면 checkpoint 학습은 건너뛰고, `RUN_TEST = True`이면 현재 checkpoint를 로드해 dev/test prediction을 생성합니다. GPU가 없는 환경에서 실수로 긴 CPU 실행이 시작되지 않도록 기본 설정에서는 CPU 실행을 막아둡니다.

In [ ]:
def require_runtime_device():
    if (RUN_TRAIN or RUN_TEST) and not args.use_gpu and not ALLOW_CPU_EXECUTION:
        raise RuntimeError(
            "GPU가 감지되지 않았습니다. GPU 없이 실행하려면 ALLOW_CPU_EXECUTION = True로 바꾸세요. "
            "단, paraphrase 모델 실행은 CPU에서 매우 오래 걸릴 수 있습니다."
        )

require_runtime_device()

if RUN_TRAIN:
    train(args)
else:
    print("Training skipped. Existing checkpoint will be used.")

if RUN_TEST:
    test(args)
else:
    print("Prediction generation skipped.")


## 8. 실행 후 prediction 요약

`test(args)` 실행 후 같은 요약 함수를 다시 호출해서 새로 생성된 prediction 파일 상태를 기록합니다. 이 출력은 제출 직전 결과 확인용으로 남깁니다.

In [ ]:
prediction_summary_after = {
    "recorded_at": datetime.now().isoformat(timespec="seconds"),
    "checkpoint": file_status(args.filepath),
    "dev": summarize_prediction_file(args.para_dev_out),
    "test": summarize_prediction_file(args.para_test_out),
}
print(json.dumps(prediction_summary_after, indent=2, ensure_ascii=False))


## 9. 실행 메모

- 실행일:
- 실행 환경:
- checkpoint:
- dev paraphrase acc:
- 특이사항:
- 다음 확인 사항:
